# Build SEMO market and operational signals

This notebook reproduces Issue #5. It downloads official SEMO publications, normalizes eleven report families, creates publication-time-safe 30/60-minute features, and inspects the quality and ablation records. Raw XML remains outside Git; processed artifacts include source URLs and SHA-256 hashes.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
PROCESSED = ROOT / 'data' / 'processed'
BENCHMARK = ROOT / 'benchmarks' / 'semo_market_signals'
ROOT

## 1. Download and build

Set `RUN_BUILD` to `True` when fresh internet access is available. `DAYS` is the number of UTC catalog dates ending today. The downloader reuses checksum-verified raw files, so interrupted builds are safe to resume.

In [ ]:
RUN_BUILD = False
DAYS = 2

if RUN_BUILD:
    command = [
        sys.executable,
        str(ROOT / 'scripts' / 'build_semo_market_data.py'),
        '--days', str(DAYS),
        '--workers', '12',
    ]
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print('Using committed processed artifacts; set RUN_BUILD=True to refresh them.')

## 2. Inspect coverage and data quality

The quality report includes all requested report families—even when the API returns no files—so an unavailable source cannot disappear silently.

In [ ]:
quality = json.loads((PROCESSED / 'semo_market_quality_report.json').read_text())
coverage = pd.DataFrame(quality['report_coverage']).T
coverage[['report_name', 'rows', 'first_publication_utc', 'last_publication_utc']]

In [ ]:
features = pd.read_csv(
    PROCESSED / 'semo_market_features_asof_30_60.csv',
    parse_dates=['issue_timestamp_utc', 'target_timestamp_utc'],
)
dictionary = pd.read_csv(PROCESSED / 'semo_market_feature_dictionary.csv')

print(f'{len(features):,} rows x {features.shape[1]:,} columns')
display(dictionary[dictionary['role'] == 'model_feature'].head(20))
display(features.head())

## 3. Recheck leakage invariants

Every target must equal issue time plus its declared horizon. Every publication or completed observation used by a row must be known at or before issue time.

In [ ]:
expected_target = features['issue_timestamp_utc'] + pd.to_timedelta(
    features['forecast_horizon_minutes'], unit='m'
)
assert (features['target_timestamp_utc'] == expected_target).all()
assert quality['duplicate_natural_keys'] == 0
assert quality['future_publication_violations'] == 0
assert quality['future_observation_violations'] == 0
print('All target-alignment and future-information checks passed.')

## 4. Read the model decision

The current retained SEMO archive does not overlap the frozen January 2026 labelled benchmark. The feature family is therefore recorded, but not promoted or assigned made-up metrics.

In [ ]:
ablation = json.loads((BENCHMARK / 'ablation_report.json').read_text())
pd.Series(ablation, name='value').to_frame()